In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.middleware import SummarizationMiddleware

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
    profile={"max_input_tokens": 1000000}
)

In [2]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("../todo_workspace")

@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"

    if not target.exists():
        return f"错误：目录不存在: {path}"

    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)

@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"

In [3]:
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich import print as rprint
agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file,run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt="你是一个代码修复助手。无论任务多简单，必须先调用 write_todos"
)


response = agent.invoke({
    "messages" : [HumanMessage("请测试并修复工作区下的my_add.py文件中的代码")]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下的my_add.py文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='20dba28c-3299-48a1-8c62-56edae5c734f'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 139,
                    'prompt_tokens': 1586,
                    'total_tokens': 1725,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 35,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '20260713153800efefc285bed04e12',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f5a69-550f-7020-a7d0-0e1f4042a289-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': '查看工作区文件结构', 'status': 'in_progress'},
                            {'content': '阅读 my_add.py 源代码', 'status': 'pending'},
                            {'content': '运行测试，查看失败原因', 'status': 'pending'},
                            {'content': '修复 my_add.py 中的代码问题', 'status': 'pending'},
                            {'content': '再次运行测试，确认修复成功', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_-7453195665171019249',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1586,
                'output_tokens': 139,
                'total_tokens': 1725,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 35}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': '查看工作区文件结构', 'status': 'in_progress'}, {'content': 
'阅读 my_add.py 源代码', 'status': 'pending'}, {'content': '运行测试，查看失败原因', 'status': 'pending'}, 
{'content': '修复 my_add.py 中的代码问题', 'status': 'pending'}, {'content': '再次运行测试，确认修复成功', 
'status': 'pending'}]",
            name='write_todos',
            id='55914c17-cd9d-4dab-8efb-12f998e466cd',
            tool_call_id='call_-7453195665171019249'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 12,
                    'prompt_tokens': 1789,
                    'total_tokens': 1801,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1536}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '202607131538087c7542814a584ff0',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f5a69-7287-7a00-8181-f87214cf8e60-0',
            tool_calls=[
                {
                    'name': 'list_files',
                    'args': {'path': '.'},
                    'id': 'call_-7453111758689926039',
       